# Faster R-CNN 海洋生物检测

在Google Colab中训练Faster R-CNN模型进行海洋生物检测任务。

**数据集信息：**
- 7个类别：fish, jellyfish, penguin, shark, puffin, stingray, starfish
- COCO格式标注
- 严重不平衡（Fish 2,669个 vs Starfish 116个）

**输出：** mAP指标和模型检查点

## 1️⃣ 安装依赖

In [ ]:
# 安装必要的包
!pip install -q torch torchvision
!pip install -q pycocotools
!pip install -q albumentations
!pip install -q pyyaml
!pip install -q tensorboard

print('✓ 依赖安装完成')

## 2️⃣ 挂载Google Drive并设置路径

In [ ]:
from google.colab import drive
import os

# 挂载Google Drive
drive.mount('/content/drive')

# 设置数据集路径（需要根据你的实际路径修改）
dataset_root = '/content/drive/MyDrive/ADL4IP_Dataset'  # ⚠️ 修改为你的数据集路径

# 验证数据集存在
if os.path.exists(dataset_root):
    print(f'✓ 数据集路径正确: {dataset_root}')
    print(f'  内容: {os.listdir(dataset_root)}')
else:
    print(f'✗ 数据集路径不存在: {dataset_root}')
    print('  请检查路径是否正确')

## 3️⃣ 克隆项目或加载代码

In [ ]:
import sys
sys.path.append('/content')

# 选择一：从GitHub克隆
!git clone https://github.com/hana240606707-bit/ADL4IP.git /content/ADL4IP
!cd /content/ADL4IP && git checkout faster-rcnn-development

sys.path.insert(0, '/content/ADL4IP')
print('✓ 项目代码加载完成')

## 4️⃣ 导入库

In [ ]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns
import json
import logging
from pathlib import Path

# 项目模块
from src.data_loader import create_data_loaders, get_class_counts
from src.model import create_model, get_model_info
from src.train import Trainer
from src.evaluate import COCOEvaluator, print_evaluation_results
from src.utils import compute_class_weights, get_logger, plot_class_distribution, ensure_dirs

# 配置
rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

# 日志
logger = get_logger('Faster_RCNN', log_file='/content/training.log')

print('✓ 所有库导入完成')
print(f'  PyTorch版本: {torch.__version__}')
print(f'  CUDA可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

## 5️⃣ 数据分析

In [ ]:
# 获取类别统计
train_ann_file = os.path.join(dataset_root, 'train', 'annotations.json')
val_ann_file = os.path.join(dataset_root, 'val', 'annotations.json')

train_class_counts = get_class_counts(train_ann_file)
val_class_counts = get_class_counts(val_ann_file)

print("\n" + "="*60)
print("训练集类别分布")
print("="*60)
total_train = sum(train_class_counts.values())
for cls, count in sorted(train_class_counts.items(), key=lambda x: x[1], reverse=True):
    pct = 100 * count / total_train
    print(f"{cls:15s}: {count:5d} ({pct:5.1f}%)")
print(f"{'总计':15s}: {total_train:5d}")

print("\n" + "="*60)
print("验证集类别分布")
print("="*60)
total_val = sum(val_class_counts.values())
for cls, count in sorted(val_class_counts.items(), key=lambda x: x[1], reverse=True):
    pct = 100 * count / total_val
    print(f"{cls:15s}: {count:5d} ({pct:5.1f}%)")
print(f"{'总计':15s}: {total_val:5d}")

# 计算不平衡比率
max_count = max(train_class_counts.values())
min_count = min(train_class_counts.values())
imbalance_ratio = max_count / min_count
print(f"\n不平衡比率: {imbalance_ratio:.1f}x（{max_count} vs {min_count}）")

## 6️⃣ 类别权重计算

In [ ]:
# 计算处理不平衡的权重
class_weights = compute_class_weights(train_class_counts)

print("\n" + "="*60)
print("类别权重（用于损失函数）")
print("="*60)
print("\n反向频率权重: weight = total / (num_classes * count)")
print("归一化范围: [1.0, max_weight]\n")

for cls, weight in sorted(class_weights.items(), key=lambda x: x[1], reverse=True):
    print(f"{cls:15s}: {weight:6.2f}x")

# 保存权重
weights_dict = {'train': class_weights, 'val': class_weights}  # 可根据val集调整
print("\n✓ 类别权重已计算，将在训练时应用")

## 7️⃣ 创建数据加载器

In [ ]:
# 创建数据加载器
train_img_dir = os.path.join(dataset_root, 'train', 'images')
val_img_dir = os.path.join(dataset_root, 'val', 'images')

logger.info("Creating data loaders...")

train_loader, val_loader = create_data_loaders(
    train_img_dir=train_img_dir,
    train_ann_file=train_ann_file,
    val_img_dir=val_img_dir,
    val_ann_file=val_ann_file,
    batch_size=4,
    num_workers=2,
    pin_memory=True
)

print(f"✓ 训练集批次数: {len(train_loader)}")
print(f"✓ 验证集批次数: {len(val_loader)}")
print(f"  批大小: 4")
print(f"  总训练样本数: {len(train_loader) * 4}")
print(f"  总验证样本数: {len(val_loader) * 4}")

## 8️⃣ 创建模型

In [ ]:
# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")

# 创建模型
# 7个类别 + 1个背景类 = 8
num_classes = 8
model = create_model(
    num_classes=num_classes,
    pretrained=True,  # 使用ImageNet预训练
    trainable_backbone_layers=3  # 冻结一些层以节省显存
)

model = model.to(device)

# 打印模型信息
info = get_model_info(model)
print(f"\n模型信息:")
print(f"  架构: Faster R-CNN + ResNet50-FPN")
print(f"  类别数: {num_classes}")
print(f"  总参数数: {info['total_parameters']:,.0f}")
print(f"  可训练参数: {info['trainable_parameters']:,.0f}")
print(f"  冻结参数: {info['frozen_parameters']:,.0f}")
print(f"  设备: {device}")

## 9️⃣ 训练配置

In [ ]:
# 创建检查点目录
checkpoint_dir = '/content/checkpoints'
ensure_dirs([checkpoint_dir])

# 创建训练器
trainer = Trainer(model, device, checkpoint_dir=checkpoint_dir)

# 设置训练参数
trainer.setup_training(
    learning_rate=0.005,
    weight_decay=0.0005,
    lr_scheduler='step',
    class_weights=class_weights  # 传入类别权重
)

print("✓ 训练配置完成")
print(f"  学习率: 0.005")
print(f"  权重衰减: 0.0005")
print(f"  LR调度器: StepLR(step_size=10, gamma=0.1)")
print(f"  类别权重: 已应用（处理不平衡）")
print(f"  检查点目录: {checkpoint_dir}")

## 🔟 开始训练

⚠️ **这一步需要时间！** 预计每个epoch 5-10分钟（取决于GPU）

In [ ]:
# 开始训练
logger.info("Starting training...")

trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=30,  # 可根据需要调整
    save_best=True
)

print("\n✓ 训练完成！")

## 1️⃣1️⃣ 加载最佳模型

In [ ]:
# 加载最佳模型
best_checkpoint = os.path.join(checkpoint_dir, 'best_model.pth')

if os.path.exists(best_checkpoint):
    trainer.load_checkpoint(best_checkpoint)
    print(f"✓ 加载最佳模型: {best_checkpoint}")
else:
    print(f"✗ 未找到最佳模型: {best_checkpoint}")

## 1️⃣2️⃣ 模型评估（mAP计算）

In [ ]:
# 创建评估器
evaluator = COCOEvaluator(
    annotation_file=val_ann_file,
    image_dir=val_img_dir
)

logger.info("Evaluating model on validation set...")

# 评估
coco_stats, per_class_stats = evaluator.evaluate(
    model=model,
    data_loader=val_loader,
    device=device,
    conf_threshold=0.5
)

# 打印结果
if coco_stats is not None:
    print_evaluation_results(coco_stats, per_class_stats, evaluator.class_names)
    
    # 保存结果
    results = {
        'coco_stats': coco_stats.tolist() if hasattr(coco_stats, 'tolist') else list(coco_stats),
        'per_class_stats': per_class_stats
    }
    
    results_file = os.path.join(checkpoint_dir, 'evaluation_results.json')
    with open(results_file, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\n✓ 结果已保存: {results_file}")
else:
    print("✗ 评估失败")

## 1️⃣3️⃣ 结果总结

In [ ]:
print("\n" + "="*70)
print("训练总结")
print("="*70)

print(f"\n数据集:")
print(f"  - 类别数: 7")
print(f"  - 训练样本: {sum(train_class_counts.values())}")
print(f"  - 验证样本: {sum(val_class_counts.values())}")
print(f"  - 不平衡比率: {imbalance_ratio:.1f}x")

print(f"\n模型:")
print(f"  - 架构: Faster R-CNN ResNet50-FPN")
print(f"  - 参数数: {info['total_parameters']:,.0f}")
print(f"  - 预训练: ImageNet")

print(f"\n处理不平衡方法:")
print(f"  1. 加权损失函数（分类损失权重 1.5x）")
print(f"  2. 数据增强（随机翻转、旋转、颜色抖动等）")
print(f"  3. 迁移学习（ImageNet预训练）")
print(f"  4. 类别权重:")
for cls, weight in sorted(class_weights.items(), key=lambda x: x[1], reverse=True):
    print(f"     - {cls}: {weight:.2f}x")

print(f"\n检查点目录: {checkpoint_dir}")
if os.path.exists(best_checkpoint):
    print(f"✓ 最佳模型已保存")
    
print("\n✓ 所有任务完成！")